In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Global top tiles per cluster (RAWH / h-space) with context windows.

What it does
------------
• Loads RAWH k-means (K=10) and a CLAM checkpoint
• Streams features from H5s, projects to h-space, assigns clusters, measures distance to centers
• Keeps the global TOP_K (smallest distance) tiles per cluster across ALL slides
• Extracts a context window (level-0) around each tile and saves as PNG
• Builds a 3×3 montage (top 9) per cluster

Assumptions
-----------
• H5 files have datasets: "features" (N, 1536) and "coords" (N, 2) as TOP-LEFT pixel coords at level 0
• Slide IDs (H5 stem) match the SVS filename stem
• Openslide is installed; SVS_ROOT contains .svs/.tif/.ndpi/mrxs

Editables
---------
• Paths, TOP_K, TILE_SIZE_L0, CONTEXT_MULT, GRID_SIZE, JPEG quality (PNG used here)
"""

from pathlib import Path
import gc, heapq
import numpy as np
import pandas as pd
import torch, h5py, joblib, openslide
from PIL import Image

# -------------------- PATHS --------------------
SVS_ROOT     = Path("/common/users/wq50/CLAM/HNSCC_slides")
FEAT_DIR     = Path("/common/users/wq50/CLAM/features/HPV_UNI2_features/h5_files")
RAWH_MODEL   = Path("/common/users/wq50/CLAM2/kmeans_models/hpv_uni2_k10_rawh.joblib")
CLAM_WEIGHT  = Path("/common/users/wq50/CLAM/results/HPV_CLAM_50_mb_s1/s_9_checkpoint.pt")
EMBED_DIM    = 1536

OUT_DIR      = Path("./global_top_tiles_rawh"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- SETTINGS --------------------
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
BATCH          = 16_384
TOP_K          = 60                 # keep this many per cluster globally
GRID_SIZE      = (3, 3)             # montage shape
TILE_SIZE_L0   = 256                # original tile edge at level 0
CONTEXT_MULT   = 2.0                # context window multiplier (e.g., 2.0 => 512x512 if tile is 256)
PNG_OPTIMIZE   = True

SVS_EXTS       = {".svs", ".tif", ".tiff", ".ndpi", ".mrxs"}

# -------------------- CLAM (h-space) --------------------
from models.model_clam import CLAM_MB

@torch.inference_mode()
def load_clam(weight_path: Path, device: str, embed_dim: int):
    m = CLAM_MB(gate=True, size_arg="small", n_classes=2, embed_dim=embed_dim)
    sd = torch.load(weight_path, map_location=device)
    m.load_state_dict(sd, strict=False)
    return m.to(device).eval()

@torch.inference_mode()
def project_h(block_np: np.ndarray, clam: CLAM_MB, device: str):
    x = torch.from_numpy(block_np).to(device)
    _, h = clam.attention_net(x)
    return h.cpu().numpy().astype(np.float32)

# -------------------- H5 & SVS helpers --------------------
def iter_h5(h5_path: Path, batch=BATCH):
    with h5py.File(h5_path, "r") as f:
        X = f["features"]; C = f["coords"]  # coords: TOP-LEFT @ LEVEL-0
        N = X.shape[0]
        for off in range(0, N, batch):
            yield off, X[off:off+batch][:].astype(np.float32), C[off:off+batch][:].astype(np.int32)

def find_svs_by_stem(root: Path, stem: str) -> Path | None:
    for p in root.rglob("*"):
        if p.suffix.lower() in SVS_EXTS and p.stem == stem:
            return p
    return None

# -------------------- Cluster assignment & ranking --------------------
def assign_and_dist_h(H: np.ndarray, km) -> tuple[np.ndarray, np.ndarray]:
    """Return labels (N,) and distances to own center (N,) in h-space."""
    labs = km.predict(H)
    C = km.cluster_centers_.astype(np.float32)
    d = np.linalg.norm(H - C[labs], axis=1)
    return labs.astype(np.int32), d.astype(np.float32)

# -------------------- Extraction with context --------------------
def extract_context_patch(svs_path: Path, x0: int, y0: int, tile_edge: int, mult: float) -> Image.Image:
    """
    x0,y0 are TOP-LEFT of the tile at level 0.
    We make a centered context window with edge = round(tile_edge * mult).
    """
    slide = openslide.OpenSlide(str(svs_path))
    W, H = slide.dimensions

    cx = x0 + tile_edge // 2
    cy = y0 + tile_edge // 2
    edge = int(round(tile_edge * mult))
    half = edge // 2

    left = int(cx - half)
    top  = int(cy - half)

    # Clamp read window inside slide bounds; pad if needed
    # OpenSlide pads OOB with black—convert to white background afterward
    region = slide.read_region((left, top), 0, (edge, edge))  # RGBA
    slide.close()

    # Convert to white background instead of black padding
    rgba = region.convert("RGBA")
    bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
    out = Image.alpha_composite(bg, rgba).convert("RGB")
    return out

# -------------------- Grid montage --------------------
def save_grid_3x3(images: list[Image.Image], out_path: Path, pad: int = 8, bg=(255,255,255)):
    if not images:
        return
    # Make all same size
    w = min(img.width for img in images)
    h = min(img.height for img in images)
    imgs = [img.resize((w,h), Image.BILINEAR) for img in images[:9]]

    rows, cols = GRID_SIZE
    canvas = Image.new("RGB", (cols*w + (cols+1)*pad, rows*h + (rows+1)*pad), bg)
    k = 0
    for r in range(rows):
        for c in range(cols):
            if k >= len(imgs): break
            x = pad + c*(w + pad)
            y = pad + r*(h + pad)
            canvas.paste(imgs[k], (x,y))
            k += 1
    canvas.save(out_path, format="PNG", optimize=PNG_OPTIMIZE)

# -------------------- Main --------------------
def main():
    # load models
    km_rawh = joblib.load(RAWH_MODEL)
    assert hasattr(km_rawh, "cluster_centers_"), "Invalid RAWH KMeans model."
    K = int(km_rawh.cluster_centers_.shape[0])

    clam = load_clam(CLAM_WEIGHT, DEVICE, EMBED_DIM)

    # discover slides present in both H5 and SVS
    h5s = sorted(FEAT_DIR.glob("*.h5"))
    slides = []
    for h5 in h5s:
        sid = h5.stem
        svs = find_svs_by_stem(SVS_ROOT, sid)
        if svs is not None:
            slides.append((sid, h5, svs))
    print(f"[discover] {len(slides)} slides with features + SVS")

    # per-cluster min-heaps storing (-score, index) trick? We want smallest distance, so store -distance? Easier: max-heap of size TOP_K per cluster.
    # We'll hold for each k: a list of tuples (-d, sid, x, y) so the largest -d (i.e., smallest negative) pops last.
    heaps = [ [] for _ in range(K) ]  # max-heaps by distance via (-d)
    tile_records = [ [] for _ in range(K) ]  # (slide_id, x, y, dist)

    # Pass 1: collect global TOP_K per cluster
    for sid, h5, _ in slides:
        for _, X, C in iter_h5(h5):
            H = project_h(X, clam, DEVICE)
            labs, d = assign_and_dist_h(H, km_rawh)
            for (lab, dist, (x, y)) in zip(labs, d, C):
                heap = heaps[lab]
                item = (-float(dist), sid, int(x), int(y), float(dist))
                if len(heap) < TOP_K:
                    heapq.heappush(heap, item)
                else:
                    # if current dist is smaller (better) than worst in heap, replace
                    if -heap[0][0] > dist:
                        heapq.heapreplace(heap, item)
        gc.collect()

    # Convert heaps to sorted ascending distance lists
    for k in range(K):
        items = [(-negd, sid, x, y, dist) for (negd, sid, x, y, dist) in heaps[k]]
        # Sort by distance ASC
        items.sort(key=lambda t: t[4])
        tile_records[k] = [(sid, x, y, dist) for (_, sid, x, y, dist) in items]
        print(f"[cluster {k:02d}] kept {len(tile_records[k])} tiles")

    # Pass 2: extract and save tiles + grids
    for k in range(K):
        out_k = OUT_DIR / f"cluster_{k:02d}"
        out_k.mkdir(parents=True, exist_ok=True)
        imgs_for_grid = []
        count = 0
        for sid, x, y, dist in tile_records[k]:
            svs_path = find_svs_by_stem(SVS_ROOT, sid)
            if svs_path is None:
                continue
            img = extract_context_patch(svs_path, x, y, TILE_SIZE_L0, CONTEXT_MULT)

            fname = f"top_{count:02d}_{sid}_x{x}_y{y}_d{dist:.4f}.png"
            img.save(out_k / fname, format="PNG", optimize=PNG_OPTIMIZE)
            if count < GRID_SIZE[0]*GRID_SIZE[1]:
                imgs_for_grid.append(img.copy())
            count += 1

        # save 3x3 montage
        if imgs_for_grid:
            save_grid_3x3(imgs_for_grid, out_k / "grid_3x3.png")
        print(f"[cluster {k:02d}] saved {count} tiles and grid")

    print("Done →", OUT_DIR.resolve())

if __name__ == "__main__":
    torch.set_float32_matmul_precision("high")
    np.random.seed(0); torch.manual_seed(0)
    main()

[discover] 106 slides with features + SVS
